In [1]:
!pip install autogluon -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 12.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.1/452.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 10.3 MB/s eta 0:

In [ ]:
#!/usr/bin/env python3
"""
Regression with AutoGluon on Moltbook data.
Compatible with older AutoGluon versions (no clean_up_fits).
"""
import os
import datetime
import shutil
import time
import gc
from google.colab import drive
drive.mount('/content/drive')

# Install AutoGluon
!pip install autogluon -q

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from autogluon.tabular import TabularDataset, TabularPredictor

# ==========================================
# CONFIGURATION
# ==========================================
DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds'
DATA_PATH = f"{DESTINATION_DIR}/processed_v1_5_4_new_full.pkl"

EMBEDDING_COL = "embeddings"
TARGET_COL = "score"
RANDOM_STATE = 42
TEST_SIZE = 0.30
VAL_SIZE_FROM_TEMP = 0.50
DROP_COLS = ["safe_content", "content", "id"]

PRESETS = 'best_quality'
TIME_LIMIT = 3600          # seconds (None for unlimited)
EVAL_METRIC = 'r2'

# ==========================================
# PRE-RUN CHECK & CLEANUP
# ==========================================
print("\n" + "="*50)
print("CLEANING UP PREVIOUS STATE")
print("="*50)

# If re-running in the same session, delete any existing predictor object
try:
    if 'predictor' in globals():
        del predictor
        gc.collect()
        print("Deleted previous predictor object.")
except:
    pass

# Create a unique path with microsecond timestamp
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")
MODEL_PATH = f"/content/drive/MyDrive/autogluon_models_{timestamp}"
RESULTS_PATH = f"/content/drive/MyDrive/autogluon_results_{timestamp}"

# Ensure the directory does not exist (should not, but we delete if it does)
if os.path.exists(MODEL_PATH):
    print(f"Deleting existing directory: {MODEL_PATH}")
    shutil.rmtree(MODEL_PATH)

print(f"Model will be saved to: {MODEL_PATH}")
print(f"Results will be saved to: {RESULTS_PATH}.pkl")

# ==========================================
# Load and prepare data
# ==========================================
print("\n" + "="*50)
print("LOADING AND PREPARING DATA")
print("="*50)

print("Loading data...")
moltbook = pd.read_pickle(DATA_PATH)
print(f"Original shape: {moltbook.shape}")

# Expand embeddings
embedding_lists = moltbook[EMBEDDING_COL].values
lengths = [len(lst) for lst in embedding_lists]
if len(set(lengths)) != 1:
    raise ValueError("Embedding lists have varying lengths.")
emb_dim = lengths[0]
print(f"Embedding dimension: {emb_dim}")

emb_df = pd.DataFrame(
    np.vstack(embedding_lists),
    index=moltbook.index,
    columns=[f"emb_{i}" for i in range(emb_dim)]
)

# Base features
X_base = moltbook.drop(columns=[TARGET_COL, EMBEDDING_COL] + DROP_COLS)

# ==========================================
# CONVERT FORUM DUMMY COLUMNS TO CATEGORICAL
# ==========================================
forum_columns = ['forum_philosophy', 'forum_technology', 'forum_todayilearned']
existing_forum_cols = [col for col in forum_columns if col in X_base.columns]
if existing_forum_cols:
    X_base['forum'] = 'other'
    for col in existing_forum_cols:
        forum_name = col.replace('forum_', '')
        X_base.loc[X_base[col] == 1, 'forum'] = forum_name
    X_base = X_base.drop(columns=existing_forum_cols)

# ==========================================
# HANDLE HOUR AS CATEGORICAL
# ==========================================
if 'hour' in X_base.columns:
    X_base['hour'] = X_base['hour'].astype(int)
    X_base['hour_category'] = X_base['hour'].astype(str)
    X_base = X_base.drop(columns=['hour'])

# Identify categorical columns
categorical_cols = []
for col in X_base.columns:
    if col in ['forum', 'hour_category'] or X_base[col].dtype in ['object', 'category']:
        categorical_cols.append(col)
        X_base[col] = X_base[col].astype('category')

numerical_cols = [col for col in X_base.columns if col not in categorical_cols]

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {len(numerical_cols)}")

# ==========================================
# Target transformation
# ==========================================
y_raw = moltbook[TARGET_COL].clip(lower=0)
y = np.log1p(y_raw)

# ==========================================
# Train / Validation / Test Split
# ==========================================
X_base_train, X_base_temp, emb_train, emb_temp, y_train, y_temp = train_test_split(
    X_base, emb_df, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
X_base_val, X_base_test, emb_val, emb_test, y_val, y_test = train_test_split(
    X_base_temp, emb_temp, y_temp, test_size=VAL_SIZE_FROM_TEMP, random_state=RANDOM_STATE
)

# Reset indices
X_base_train = X_base_train.reset_index(drop=True)
X_base_val   = X_base_val.reset_index(drop=True)
X_base_test  = X_base_test.reset_index(drop=True)
emb_train    = emb_train.reset_index(drop=True)
emb_val      = emb_val.reset_index(drop=True)
emb_test     = emb_test.reset_index(drop=True)
y_train      = y_train.reset_index(drop=True)
y_val        = y_val.reset_index(drop=True)
y_test       = y_test.reset_index(drop=True)

# Combine into DataFrames
train_df = pd.concat([X_base_train, emb_train, y_train], axis=1)
val_df   = pd.concat([X_base_val,   emb_val,   y_val],   axis=1)
test_df  = pd.concat([X_base_test,  emb_test,  y_test],  axis=1)

# Convert to AutoGluon's TabularDataset
train_data = TabularDataset(train_df)
val_data   = TabularDataset(val_df)
test_data  = TabularDataset(test_df)

print(f"Training data: {train_data.shape}")
print(f"Validation data: {val_data.shape}")
print(f"Test data: {test_data.shape}")

# ==========================================
# Train AutoGluon Predictor (Fresh Instance)
# ==========================================
print("\n" + "="*50)
print("TRAINING AUTOGLUON PREDICTOR")
print("="*50)

# Create a completely fresh predictor instance
predictor = TabularPredictor(
    label=TARGET_COL,
    problem_type='regression',
    eval_metric=EVAL_METRIC,
    path=MODEL_PATH
)

# Fit the model (without clean_up_fits)
predictor.fit(
    train_data=train_data,
    tuning_data=val_data,
    presets=PRESETS,
    time_limit=TIME_LIMIT,
    verbosity=2,
    use_bag_holdout=True,          # allows tuning_data with bagging
    dynamic_stacking=False,        # Disable DyStack to avoid internal sub‑fit bug
    num_stack_levels=1,            # Use 1 level of stacking
    holdout_frac=None
)

# ==========================================
# Evaluate on test set
# ==========================================
print("\n" + "="*50)
print("EVALUATION ON TEST SET")
print("="*50)

leaderboard = predictor.leaderboard(test_data, silent=True)
print(leaderboard)

test_preds = predictor.predict(test_data, model=predictor.model_best)
test_true  = test_data[TARGET_COL].values

test_r2   = r2_score(test_true, test_preds)
test_rmse = np.sqrt(mean_squared_error(test_true, test_preds))
test_mae  = mean_absolute_error(test_true, test_preds)

print(f"\nTest R² (log space): {test_r2:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE: {test_mae:.4f}")

# Original space
test_preds_orig = np.expm1(test_preds)
test_true_orig  = np.expm1(test_true)
test_r2_orig    = r2_score(test_true_orig, test_preds_orig)
test_rmse_orig  = np.sqrt(mean_squared_error(test_true_orig, test_preds_orig))
test_mae_orig   = mean_absolute_error(test_true_orig, test_preds_orig)

print(f"\nTest R² (original): {test_r2_orig:.4f}")
print(f"Test RMSE: {test_rmse_orig:.2f}")
print(f"Test MAE: {test_mae_orig:.2f}")

# # ==========================================
# # Save results
# # ==========================================
# results = {
#     'test_log': {'r2': test_r2, 'rmse': test_rmse, 'mae': test_mae},
#     'test_orig': {'r2': test_r2_orig, 'rmse': test_rmse_orig, 'mae': test_mae_orig},
#     'best_model': predictor.model_best,
#     'model_path': MODEL_PATH,
#     'leaderboard': leaderboard.to_dict()
# }
# joblib.dump(results, f"{RESULTS_PATH}.pkl")
# print(f"\nResults saved to {RESULTS_PATH}.pkl")

# # ==========================================
# # Feature importance plot
# # ==========================================
# feature_importance = predictor.feature_importance(val_data, silent=True)
# plt.figure(figsize=(10, 8))
# feature_importance.head(20).plot(kind='barh')
# plt.title('Top 20 Feature Importance')
# plt.tight_layout()
# plt.savefig(f"{MODEL_PATH}/feature_importance.png", dpi=150)
# plt.show()

# print("\n Training completed successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

CLEANING UP PREVIOUS STATE
Deleted previous predictor object.
Model will be saved to: /content/drive/MyDrive/autogluon_models_20260418_073509_944399
Results will be saved to: /content/drive/MyDrive/autogluon_results_20260418_073509_944399.pkl

LOADING AND PREPARING DATA
Loading data...
Original shape: (20287, 20)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 39.49/39.49 GB
Total GPU Memory:   Free: 39.49 GB, Allocated: 0.00 GB, Total: 39.49 GB
GPU Count:          1
Memory Avail:       80.23 GB / 83.47 GB (96.1%)
Disk Space Avail:   61.47 GB / 112.64 GB (54.6%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1


Embedding dimension: 768
Categorical columns: ['forum', 'hour_category']
Numerical columns: 11
Training data: (14200, 782)
Validation data: (3043, 782)
Test data: (3044, 782)

TRAINING AUTOGLUON PREDICTOR


Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "/content/drive/MyDrive/autogluon_models_20260418_073509_944399"
Train Data Rows:    14200
Train Data Columns: 781
Tuning Data Rows:    3043
Tuning Data Columns: 781
Label Column:       score
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    82112.48 MB
	Train Data (Original)  Memory Usage: 52.00 MB (0.1% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGener


EVALUATION ON TEST SET
                     model  score_test  score_val eval_metric  pred_time_test  \
0           XGBoost_BAG_L2    0.402497   0.403810          r2        7.659926   
1      WeightedEnsemble_L3    0.402293   0.405219          r2        7.864873   
2        LightGBMXT_BAG_L2    0.398603   0.403912          r2        6.793870   
3          LightGBM_BAG_L2    0.396020   0.400171          r2        6.804433   
4          CatBoost_BAG_L2    0.391053   0.397423          r2        6.762654   
5      WeightedEnsemble_L2    0.389892   0.390142          r2        1.690112   
6     ExtraTreesMSE_BAG_L2    0.386544   0.389682          r2        6.923973   
7   NeuralNetFastAI_BAG_L2    0.386129   0.387131          r2        7.469280   
8   NeuralNetFastAI_BAG_L1    0.384922   0.375866          r2        0.817906   
9   RandomForestMSE_BAG_L2    0.380576   0.386649          r2        6.859189   
10       LightGBMXT_BAG_L1    0.367169   0.376763          r2        0.611592   
11  

In [ ]:
!pip install --upgrade autogluon
